In [7]:
!pip install ultralytics transformers torch opencv-python --quiet

In [8]:
import cv2
import torch
import json
from ultralytics import YOLO
from transformers import AutoConfig, AutoImageProcessor, SiglipForImageClassification

In [9]:
sign_labels = {
    0: "Hello",
    1: "Thank You",
    2: "Yes",
    3: "No",
    4: "I Love You",
    5: "Good Morning",
    6: "Good Night",
    7: "Help",
    8: "Sorry",
    9: "Please",
    10: "How Are You",
    11: "I Am Fine",
    12: "Stop",
    13: "Come Here",
    14: "Go There",
    15: "Wait",
    16: "Name",
    17: "What",
    18: "Where",
    19: "When",
    20: "Why"
}

# Save to JSON so you can modify later without editing code
with open("sign_labels.json", "w") as f:
    json.dump(sign_labels, f, indent=4)

print("✅ sign_labels.json created successfully!")


✅ sign_labels.json created successfully!


In [10]:
class ModelSelector:
    def __init__(self, model_type):
        self.model_type = model_type

        if self.model_type == "sign":
            self._load_sign_model()
        elif self.model_type == "emotion":
            self._load_emotion_model()
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")

    # -----------------------------
    # Load Sign Language Model
    # -----------------------------
    def _load_sign_model(self):
        print("🔹 Loading Sign Language model...")
        try:
            config = AutoConfig.from_pretrained("sign_model")
            self.model = SiglipForImageClassification(config)

            state_dict = torch.load(
                r"C:\Users\anike\OneDrive\Desktop\ds project\HumanPatterns\sign_model\model_weights.pt",
                map_location="cpu"
            )
            self.model.load_state_dict(state_dict, strict=False)
            self.model.eval()

            self.processor = AutoImageProcessor.from_pretrained("sign_model", use_fast=False)

            # ✅ Load labels dynamically from JSON file
            try:
                with open("sign_labels.json", "r") as f:
                    self.sign_labels = json.load(f)
                print(f"✅ Loaded {len(self.sign_labels)} sign labels")
            except FileNotFoundError:
                print("⚠️ sign_labels.json not found, using numeric outputs")
                self.sign_labels = {}

            print("✅ Sign model loaded successfully")
        except Exception as e:
            print(f"❌ Failed to load Sign model: {e}")

    # -----------------------------
    # Load Emotion Model (YOLO)
    # -----------------------------
    def _load_emotion_model(self):
        print("🔹 Loading Emotion model (YOLO)...")
        try:
            self.model = YOLO(r"C:\Users\anike\OneDrive\Desktop\ds project\HumanPatterns\best.pt")
            print("✅ Emotion model loaded successfully (YOLO)")
        except Exception as e:
            print(f"❌ Failed to load Emotion model: {e}")

    # -----------------------------
    # Prediction Function
    # -----------------------------
    def predict(self, image):
        if self.model_type == "sign":
            try:
                inputs = self.processor(images=image, return_tensors="pt")
                with torch.no_grad():
                    outputs = self.model(**inputs)
                    preds = outputs.logits.argmax(-1).item()

                # ✅ Get readable label
                pred_label = self.sign_labels.get(str(preds), self.sign_labels.get(preds, str(preds)))
                return pred_label
            except Exception as e:
                print(f"⚠️ Sign Prediction Error: {e}")
                return "Unknown"

        elif self.model_type == "emotion":
            try:
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                results = self.model(image_rgb)

                # ✅ Works for both detection & classification YOLO models
                if results and len(results[0].boxes) > 0:
                    class_id = int(results[0].boxes.cls[0].item())
                    pred_name = results[0].names[class_id]
                elif results and hasattr(results[0], "probs") and results[0].probs is not None:
                    class_id = results[0].probs.argmax().item()
                    pred_name = results[0].names[class_id]
                else:
                    pred_name = "Unknown"

                return pred_name
            except Exception as e:
                print(f"⚠️ Emotion Prediction Error: {e}")
                return "Unknown"


In [11]:
class RAGRouter:
    def __init__(self):
        print("🚀 Initializing RAG Router...")
        self.sign_selector = ModelSelector("sign")
        self.emotion_selector = ModelSelector("emotion")
        print("✅ Models ready!")

    # -----------------------------
    # Run Real-Time Webcam Detection
    # -----------------------------
    def run_webcam(self):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("⚠️ Cannot open webcam.")
            return

        print("🎥 Webcam started. Press 'q' to quit.")
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            resized = cv2.resize(frame, (224, 224))

            sign_pred = self.sign_selector.predict(resized)
            emotion_pred = self.emotion_selector.predict(resized)

            cv2.putText(frame, f"Sign: {sign_pred}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(frame, f"Emotion: {emotion_pred}", (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

            cv2.imshow("🧠 RAG Real-time Detection", frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        print("👋 Webcam closed.")


In [12]:
if __name__ == "__main__":
    router = RAGRouter()
    router.run_webcam()

🚀 Initializing RAG Router...
🔹 Loading Sign Language model...
✅ Loaded 21 sign labels
✅ Sign model loaded successfully
🔹 Loading Emotion model (YOLO)...
✅ Emotion model loaded successfully (YOLO)
✅ Models ready!
🎥 Webcam started. Press 'q' to quit.

0: 640x640 (no detections), 148.9ms
Speed: 24.6ms preprocess, 148.9ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 129.9ms
Speed: 2.2ms preprocess, 129.9ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 115.4ms
Speed: 2.8ms preprocess, 115.4ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 115.5ms
Speed: 2.4ms preprocess, 115.5ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 121.6ms
Speed: 5.3ms preprocess, 121.6ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 305.1ms
Speed: 11.0ms preprocess, 305.1ms infe